# Student exercises 4D — K-means and PCA

Fill five TODOs with one or two lines, replacing each `raise NotImplementedError`. Run from the top;
data and checks are supplied, with no download.

**K-means** minimizes within-group squared distances (inertia); IDs have no fixed meaning.
**PCA** learns orthogonal high-variance directions after centering. Scores locate rows along directions;
reconstruction maps scores back to features. Here MSE averages squared error over all entries.

`fit` learns; K-means `predict` assigns to frozen centers; PCA `transform` scores using frozen means/directions,
and `inverse_transform` reconstructs. Neither model uses target labels. Fix settings before inspecting held-out rows.
See [lecture](04_clustering_and_pca.pdf); consult the separate answer key after solving.

## 1. Group two-dimensional observations

`make_blobs` creates three compact clouds; discard generating labels.
`train_test_split` reserves 80% fitting/20% test rows. Features share units, so use them directly.
Fit `KMeans` with K=3, 20 restarts, seed 23; assign test rows to its centers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

X, _ = make_blobs(n_samples=300, centers=[[-3, 0], [3, 0], [0, 4]], cluster_std=.6, random_state=41)
X_train, X_test = train_test_split(X, test_size=.2, random_state=23)
print("K-means fitting/test shapes:", X_train.shape, X_test.shape)

In [ ]:
# TODO 1 (two lines): create KMeans with n_clusters=3, n_init=20, random_state=23.
# Store it in cluster_model, then call cluster_model.fit(X_train).
raise NotImplementedError("TODO 1: create and fit K-means")

In [ ]:
# TODO 2 (one line): use cluster_model.predict(X_test); store in test_groups.
raise NotImplementedError("TODO 2: assign new rows")

The check recomputes test squared distances and nearest-center assignments.
`inertia_` uses fitting rows, unlike mean test distance.
**Explain:** Why can cluster IDs swap harmlessly? Why cannot minimum training inertia choose K?

In [ ]:
centers = cluster_model.cluster_centers_
squared_distances = np.sum((X_test[:, None, :] - centers[None, :, :])**2, axis=2)
assert np.array_equal(test_groups, squared_distances.argmin(axis=1))
print("Training inertia:", cluster_model.inertia_)
print("Test mean squared distance:", squared_distances.min(axis=1).mean())
plt.figure(figsize=(6, 4))
plt.scatter(X_test[:, 0], X_test[:, 1], c=test_groups, cmap="tab10", s=25)
plt.scatter(centers[:, 0], centers[:, 1], c="black", marker="x", s=100)
plt.xlabel("x1"); plt.ylabel("x2"); plt.title("Test assignments to fitted centers")
plt.tight_layout(); plt.show()

## 2. Compress three features to one component

Generate noisy rows near direction (1,2,−1), split 80/20, and fit one-component `PCA`.
Transform and reconstruct test rows. Keep original units (equal simulated noise scale);
`svd_solver="full"` fixes the decomposition method.

In [ ]:
rng = np.random.default_rng(42)
coordinate = rng.normal(0, 2, size=(300, 1))
X_pca = coordinate * np.array([1, 2, -1]) + rng.normal(0, .2, size=(300, 3))
X_pca_train, X_pca_test = train_test_split(X_pca, test_size=.2, random_state=23)
print("PCA fitting/test shapes:", X_pca_train.shape, X_pca_test.shape)

In [ ]:
# TODO 3 (two lines): create PCA(n_components=1, svd_solver="full") in pca_model.
# Then fit pca_model on X_pca_train.
raise NotImplementedError("TODO 3: create and fit PCA")

In [ ]:
# TODO 4 (one line): call pca_model.transform(X_pca_test); store in test_scores.
raise NotImplementedError("TODO 4: compute scores")

In [ ]:
# TODO 5 (one line): call pca_model.inverse_transform(test_scores); store in reconstructed.
raise NotImplementedError("TODO 5: reconstruct test rows")

Compare test reconstruction error with the fitting-mean baseline.
**Explain:** What shapes should `test_scores` and `reconstructed` have? Why do score-sign changes preserve reconstruction?
Does high explained variance guarantee usefulness for every prediction task?

In [ ]:
assert test_scores.shape == (len(X_pca_test), 1)
assert reconstructed.shape == X_pca_test.shape
manual = pca_model.mean_ + test_scores @ pca_model.components_
np.testing.assert_allclose(reconstructed, manual)
print("First explained variance ratio:", pca_model.explained_variance_ratio_[0])
print("Test reconstruction MSE per entry:", np.mean((X_pca_test-reconstructed)**2))
print("Mean-only test MSE per entry:", np.mean((X_pca_test-pca_model.mean_)**2))

Documentation: [KMeans](https://scikit-learn.org/1.8/modules/generated/sklearn.cluster.KMeans.html)
and [PCA](https://scikit-learn.org/1.8/modules/generated/sklearn.decomposition.PCA.html).
Look up parameters (constructor choices), attributes (learned values, often ending in `_`), and methods (operations) separately.